# Inspección de un circuito

Dibuja un circuito algorítmico con etiquetas por puerta (F7), su DAG de puertas (F4) coloreado por el bloque de origen de cada puerta y su grafo de interacción de cúbits (F5).

Variables de entorno opcionales: `QCCLASS_CONFIG` (configuración; por defecto `../configs/dataset.yaml`) y `QCCLASS_ALGORITMO` (algoritmo a inspeccionar; por defecto `grover`). Requiere haber ejecutado `qcclass formatos`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx

from qcclass.circuitos import qpy_path, read_qpy
from qcclass.config import load_dataset_config
from qcclass.formatos.construir import DAG_DIR, QUBIT_DIR
from qcclass.formatos.grafo_dag import load_graph
from qcclass.metadatos import METADATA_COLUMNS, read_table

config = load_dataset_config(Path(os.environ.get('QCCLASS_CONFIG', '../configs/dataset.yaml')))
metadata = read_table(config.rutas.metadata, METADATA_COLUMNS).set_index('id')
algorithm = os.environ.get('QCCLASS_ALGORITMO', 'grover')

## Circuito elegido

El circuito etiquetado más pequeño del algoritmo con al menos 3 cúbits.

In [ ]:
labelled = metadata[(metadata['variante_comp'] == 'etiquetada') & (metadata['algoritmo'] == algorithm)]
labelled = labelled.assign(puertas=labelled['n_puertas'].astype(int), cubits=labelled['n_qubits'].astype(int))
candidates = labelled[labelled['cubits'] >= 3] if (labelled['cubits'] >= 3).any() else labelled
circuit_id = candidates.sort_values(['puertas', 'cubits']).index[0]
gate_labels = json.loads((config.rutas.gate_labels / f'{circuit_id}.json').read_text(encoding='utf-8'))
raw_row = metadata.loc[gate_labels['id_raw']]
print(circuit_id, '| clase:', metadata.loc[circuit_id, 'clase'], '| bloques:', gate_labels['bloques'])

In [ ]:
algorithmic = read_qpy(qpy_path(config.rutas.raiz / raw_row['ruta_raw']))
algorithmic.draw('mpl', fold=-1)

## DAG de puertas (F4) coloreado por bloque (F7)

Eje horizontal: capa ASAP; eje vertical: cúbit (media de control y destino en las CX).

In [ ]:
dag = load_graph(config.rutas.graphs / DAG_DIR / f'{circuit_id}.pt')
labels = gate_labels['etiquetas']
assert dag.num_nodes == len(labels)
graph = nx.DiGraph()
graph.add_nodes_from(range(dag.num_nodes))
graph.add_edges_from(dag.edge_index.t().tolist())
layer, qubit_a, qubit_b = dag.x[:, 6], dag.x[:, 7], dag.x[:, 8]
n_qubits = int(raw_row['n_qubits'])
positions = {i: (float(layer[i]), -float(qubit_a[i] + qubit_b[i]) * n_qubits / 2) for i in graph.nodes}
blocks = sorted(set(labels), key=labels.index)
palette = plt.get_cmap('tab10')
colors = [palette(blocks.index(label) % 10) for label in labels]
fig, ax = plt.subplots(figsize=(14, 4))
nx.draw_networkx(graph, positions, node_color=colors, node_size=40, with_labels=False, arrowsize=6, width=0.4, ax=ax)
for block in blocks:
    ax.scatter([], [], color=palette(blocks.index(block) % 10), label=block)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'{circuit_id}: {dag.num_nodes} puertas')
plt.show()

## Grafo de interacción de cúbits (F5)

Grosor de arista: número de CX; etiqueta de nodo: puertas de un cúbit.

In [ ]:
qubits = load_graph(config.rutas.graphs / QUBIT_DIR / f'{circuit_id}.pt')
interaction = nx.Graph()
interaction.add_nodes_from(range(qubits.num_nodes))
for (a, b), weight in zip(qubits.edge_index.t().tolist(), qubits.edge_weight.tolist(), strict=True):
    interaction.add_edge(a, b, weight=weight)
widths = [0.5 + interaction[a][b]['weight'] / 10 for a, b in interaction.edges]
node_labels = {q: f'q{q}\n{int(qubits.x[q, 0])}' for q in interaction.nodes}
fig, ax = plt.subplots(figsize=(5, 5))
nx.draw_networkx(interaction, nx.circular_layout(interaction), labels=node_labels, width=widths, node_color='#9ecae1', node_size=900, font_size=8, ax=ax)
ax.set_axis_off()
plt.show()